In [78]:
import pandas as pd
import requests as re
from bs4 import BeautifulSoup as bs
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [79]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

In [80]:
url = "https://editorial.rottentomatoes.com/guide/popular-movies/"

In [81]:
html = re.get(url, headers=headers).text
soup = bs(html, "html.parser")

movie_links = []

for a in soup.find_all("a", href=True):
    href = a['href']

    if href.startswith("https://www.rottentomatoes.com/m/"):
        movie_links.append(href)

In [82]:
print(f"Found {len(movie_links)} movie links.")

Found 87 movie links.


In [83]:
set(movie_links)  # Remove duplicates if any

{'https://www.rottentomatoes.com/m/apex_2026',
 'https://www.rottentomatoes.com/m/backrooms',
 'https://www.rottentomatoes.com/m/dead_mans_wire',
 'https://www.rottentomatoes.com/m/disclosure_day',
 'https://www.rottentomatoes.com/m/exit_8_2025',
 'https://www.rottentomatoes.com/m/hokum',
 'https://www.rottentomatoes.com/m/hoppers',
 'https://www.rottentomatoes.com/m/i_swear_2025',
 'https://www.rottentomatoes.com/m/in_the_grey',
 'https://www.rottentomatoes.com/m/iron_lung',
 'https://www.rottentomatoes.com/m/is_god_is',
 'https://www.rottentomatoes.com/m/ladies_first_2026',
 'https://www.rottentomatoes.com/m/masters_of_the_universe_2026',
 'https://www.rottentomatoes.com/m/michael',
 'https://www.rottentomatoes.com/m/obsession_2025',
 'https://www.rottentomatoes.com/m/office_romance',
 'https://www.rottentomatoes.com/m/passenger_2026',
 'https://www.rottentomatoes.com/m/power_ballad',
 'https://www.rottentomatoes.com/m/pressure_2026',
 'https://www.rottentomatoes.com/m/project_hail_m

In [84]:
all_reviews = []

In [85]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [86]:
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

In [87]:
cards = driver.find_elements(By.CSS_SELECTOR, "review-card-audience")

for card in cards:

    try:
        review = card.find_element(
            By.CSS_SELECTOR,
            'span[slot="content"]'
        )

        print(review.text)

    except Exception as e:
        print(e)

In [88]:
# =====================================
# LOOP THROUGH MOVIES
# =====================================

for movie_url in movie_links:

    movie_name = movie_url.rstrip("/").split("/")[-1]

    review_url = movie_url.rstrip("/") + "/reviews/all-audience"

    print(f"\nScraping: {movie_name}")

    try:

        driver.get(review_url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "review-card-audience")
            )
        )

        time.sleep(3)

        # =====================================
        # CLICK LOAD MORE UNTIL FINISHED
        # =====================================

        while True:

            try:

                load_more = driver.find_element(
                    By.CSS_SELECTOR,
                    'rt-button[data-qa="load-more-btn"]'
                )

                driver.execute_script(
                    "arguments[0].click();",
                    load_more
                )

                print("Loading more reviews...")

                time.sleep(2)

            except:
                break

        # =====================================
        # GET REVIEW CARDS
        # =====================================

        cards = driver.find_elements(
            By.CSS_SELECTOR,
            "review-card-audience"
        )

        print(f"Found {len(cards)} review cards")

        movie_count = 0

        # =====================================
        # EXTRACT REVIEWS
        # =====================================

        for card in cards:

            try:

                review_text = card.find_element(
                    By.CSS_SELECTOR,
                    'span[slot="content"]'
                ).text.strip()

                if review_text:

                    all_reviews.append({
                        "movie": movie_name,
                        "review": review_text
                    })

                    movie_count += 1

            except:
                continue

        print(f"Successfully extracted {movie_count} reviews")

    except Exception as e:

        print(f"Error scraping {movie_name}: {e}")

# =====================================
# CLOSE BROWSER
# =====================================

driver.quit()


Scraping: scary_movie_2026
Found 10 review cards
Successfully extracted 10 reviews

Scraping: scary_movie_2026
Found 10 review cards
Successfully extracted 10 reviews

Scraping: masters_of_the_universe_2026
Found 10 review cards
Successfully extracted 10 reviews

Scraping: masters_of_the_universe_2026
Found 10 review cards
Successfully extracted 10 reviews

Scraping: masters_of_the_universe_2026
Found 10 review cards
Successfully extracted 10 reviews

Scraping: backrooms
Found 10 review cards
Successfully extracted 10 reviews

Scraping: backrooms
Found 10 review cards
Successfully extracted 10 reviews

Scraping: obsession_2025
Found 10 review cards
Successfully extracted 10 reviews

Scraping: obsession_2025
Found 10 review cards
Successfully extracted 10 reviews

Scraping: obsession_2025
Found 10 review cards
Successfully extracted 10 reviews

Scraping: hokum
Found 10 review cards
Successfully extracted 10 reviews

Scraping: hokum
Found 10 review cards
Successfully extracted 10 review

In [89]:
# 

In [90]:
df = pd.DataFrame(all_reviews)

In [91]:
print(f"\nTotal reviews scraped: {len(df)}")


Total reviews scraped: 870


In [93]:
df.to_csv("movie_reviews.csv", index=False)